### Pré-traitement

In [14]:
# Import des librairies
import pandas as pd
import numpy as np
import hashlib
from rapidfuzz import process, fuzz
import re

In [15]:
# On récupère les données de stats
data_ids = pd.read_csv("../data/entity_resolution.csv")
data_opta_analyst = pd.read_csv("../data/silver_analyst.csv")
data_fotmob = pd.read_csv("../data/fotmob_v2.csv")
data_sofascore = pd.read_csv("../data/sofascore_v2.csv")
data_understat = pd.read_csv("../data/silver_understat.csv")
data_jugadores = pd.read_csv("../data/jugadores.csv")

In [16]:
# On enlève les lignes où nationality est vide
#data_jugadores = data_jugadores[data_jugadores["nationality"].notna() & (data_jugadores["nationality"].str.strip() != "")].copy()

# On comptabilise le nombre d'identifiants disponible pour un joueur sur les différents fournisseurs de données

cols_ids = ["fotmob_id","sofascore_id","tm_id"]

n_complet = data_jugadores[cols_ids].notna().all(axis=1).sum()

print(n_complet)

n_total = len(data_jugadores)
pct_complet = n_complet / n_total * 100

print(f"{n_complet} lignes sur {n_total} ({pct_complet:.1f} %)")

7749
7749 lignes sur 20080 (38.6 %)


### Opta

In [17]:
# On enlève les joueurs ayant participé à moins de 90 minute dans une compétition
data_opta_analyst = data_opta_analyst[data_opta_analyst["minutes"] >= 90].copy()

# On garde que les minutes jouées dans le big 5 + compétitions européennes
top_leagues = ["Premier League","Serie A","La Liga","Ligue 1","Bundesliga","Champions League","Europa League","Conference League"]

# On enlève les joueurs problématiques au niveau des homonymes
#data_opta_analyst = data_opta_analyst[
    #~data_opta_analyst["name"].isin(["Nico González","Vitinha","Weysley","Idrissa Gueye"])
#].copy()

data_opta_analyst = data_opta_analyst[data_opta_analyst["league"].isin(top_leagues)].copy()

# Liste des joueurs homonymes à traiter séparément
homonymes = ["Nico González","Vitinha","Idrissa Gueye"]

# On ajoute l'identifiant Transfermarkt pour les joueurs sans homonymie
jugadores_merge = data_jugadores[~data_jugadores["analyst_name"].isin(homonymes)][["analyst_name","tm_id"]].drop_duplicates()

data_opta_analyst = data_opta_analyst.merge(jugadores_merge,
    left_on="name",right_on="analyst_name",how="left")

# Identifiants Transfermarkt des joueurs homonymes selon leur équipe
manual_tm_ids = {
    ("Nico González", "7vn2i2kd35zuetw6b38gw9jsz"): 466805, ("Nico González", "a3nyxabgsqlnqfkeg41m6tnpp"): 466805,
    ("Nico González", "bqbbqm98ud8obe45ds9ohgyrd"): 486031, ("Nico González", "4ku8o6uf87yd8iecdalipo6wd"): 486031,
    ("Vitinha", "2b3mar72yy8d6uvat1ka6tn3r"): 487469, ("Idrissa Gueye", "dxq76zcvnokq07cszdx0i6kve"): 1178488,
    ("Idrissa Gueye", "2khen2a38l2hkx33s73pehl6o"): 1178488, ("Idrissa Gueye", "ehd2iemqmschhj2ec0vayztzz"): 126665
}

# On attribue le bon tm_id aux homonymes
for (name, team_id), tm_id in manual_tm_ids.items():
    data_opta_analyst.loc[(data_opta_analyst["name"] == name) & (data_opta_analyst["team_id"] == team_id),"tm_id"] = tm_id

# Suppression de la colonne analyst_name ajoutée par le merge
data_opta_analyst = data_opta_analyst.drop(columns="analyst_name")

# On garde uniquement les lignes ayant tm_id
data_opta_analyst = data_opta_analyst[data_opta_analyst["tm_id"].notna()].copy()

# Variables à sommer
sum_cols = ["apps","minutes","atk_goals","atk_xg","atk_goals_vs_xg","atk_shots","atk_shots_on_target","def_tackles",
    "def_interceptions","def_possession_won","def_blocks","def_clearances","def_ground_duels_total",
    "def_ground_duels_won","def_aerial_duels_total","def_aerial_duels_won","pass_total","pass_open_play_total",
    "pass_final_third","pass_crosses","pass_long_total","pass_through_balls","carry_all_carries","carry_progressive",
    "carry_distance_m","carry_prog_distance_m","carry_lead_to_shot","carry_lead_to_goal","carry_lead_to_chance","carry_lead_to_assist",
    "gk_goals_conceded","gk_saves","gk_xgot_conceded","gk_goals_prevented"]

# Variables à moyenner en fonction des minutes
weighted_cols = ["atk_conversion_pct","atk_xg_per_shot","def_ground_duels_pct","def_aerial_duels_pct",
    "pass_accuracy_pct","pass_open_play_pct","pass_long_pct","carry_avg_distance_m","carry_prog_avg_distance_m","gk_save_pct"]


# Variables pour lesquelles on créera un per90
per90_cols = ["atk_goals","atk_xg","atk_shots","atk_shots_on_target","def_tackles","def_interceptions","def_possession_won",
    "def_blocks","def_clearances","def_ground_duels_total","def_ground_duels_won","def_aerial_duels_total",
    "def_aerial_duels_won","pass_total","pass_open_play_total","pass_final_third","pass_crosses","pass_long_total",
    "pass_through_balls","carry_all_carries","carry_progressive","carry_distance_m","carry_prog_distance_m",
    "carry_lead_to_shot","carry_lead_to_goal","carry_lead_to_chance","carry_lead_to_assist",
    "gk_goals_conceded","gk_saves","gk_xgot_conceded","gk_goals_prevented"]


# On ne garde que les colonnes réellement présentes
sum_cols = [
    col for col in sum_cols
    if col in data_opta_analyst.columns
]

weighted_cols = [
    col for col in weighted_cols
    if col in data_opta_analyst.columns
]

per90_cols = [
    col for col in per90_cols
    if col in data_opta_analyst.columns
]

def aggregate_player_opta(group):

    result = {}

    # group.name = (tm_id, season)
    tm_id, season = group.name

    result["tm_id"] = tm_id
    result["season"] = season

    # League : concatène uniquement les valeurs différentes
    result["league"] = ", ".join(
        group["league"]
        .dropna()
        .astype(str)
        .drop_duplicates()
    )

    # Sommes
    for col in sum_cols:
        result[col] = group[col].sum(min_count=1)

    # Moyennes pondérées par les minutes
    for col in weighted_cols:

        mask = (group[col].notna() & group["minutes"].notna() & (group["minutes"] > 0))

        if mask.any():
            result[col] = np.average(
                group.loc[mask, col],
                weights=group.loc[mask, "minutes"]
            )
        else:
            result[col] = np.nan

    # Autres variables : première valeur disponible
    processed_cols = (
        {"tm_id", "season", "league"}
        | set(sum_cols)
        | set(weighted_cols)
    )

    other_cols = [
        col for col in group.columns
        if col not in processed_cols
    ]

    for col in other_cols:

        values = group[col].dropna()

        result[col] = (
            values.iloc[0]
            if len(values) > 0
            else np.nan
        )

    return pd.Series(result)


# Agrégation par joueur ET par saison
data_opta_analyst_agg = (
    data_opta_analyst
    .groupby(
        ["tm_id", "season"],
        dropna=False
    )
    .apply(aggregate_player_opta)
    .reset_index(drop=True)
)


# Création des variables /90
for col in per90_cols:

    data_opta_analyst_agg[f"{col}_per90"] = np.where(data_opta_analyst_agg["minutes"] > 0,
        data_opta_analyst_agg[col] / data_opta_analyst_agg["minutes"] * 90, np.nan)


# Mettre name en première colonne
cols = ["name"] + [
    col for col in data_opta_analyst_agg.columns
    if col != "name"
]

data_opta_analyst_agg = data_opta_analyst_agg[cols]

# On enlève les joueurs ayant participé à moins de 270 minutes en cummulé sur plusieurs compétitions sur une saison
data_opta_analyst_agg = data_opta_analyst_agg[data_opta_analyst_agg["minutes"] >= 270].copy()

# On arrondit toutes les variables numériques à 2 chiffres après la virgule
cols_numeric = data_opta_analyst_agg.select_dtypes(include="number").columns

data_opta_analyst_agg[cols_numeric] = data_opta_analyst_agg[cols_numeric].round(2)

# Liste des big_5 leagues
big_5_leagues = ["Premier League","Serie A","La Liga","Ligue 1","Bundesliga"]

data_complete = data_opta_analyst_agg[
    data_opta_analyst_agg["league"].apply(
        lambda x: any(league in x.split(", ") for league in big_5_leagues)
    )
].copy()

In [18]:
data_complete

,name,tm_id,season,league,apps,minutes,atk_goals,atk_xg,atk_goals_vs_xg,atk_shots,...,carry_distance_m_per90,carry_prog_distance_m_per90,carry_lead_to_shot_per90,carry_lead_to_goal_per90,carry_lead_to_chance_per90,carry_lead_to_assist_per90,gk_goals_conceded_per90,gk_saves_per90,gk_xgot_conceded_per90,gk_goals_prevented_per90
0,Emerson,70.0,2025/2026,"Ligue 1, Champions League",33,2504,0.0,0.58,-0.58,15.0,...,156.39,83.63,0.14,0.00,0.32,0.11,NaN,NaN,NaN,NaN
3,James Milner,3333.0,2025/2026,Premier League,20,778,1.0,1.02,-0.02,8.0,...,80.68,39.01,0.12,0.00,0.23,0.00,NaN,NaN,NaN,NaN
6,Abdoulaye Faye,6107.0,2025/2026,Ligue 1,17,1446,0.0,0.59,-0.59,8.0,...,110.06,46.18,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN
7,Jonas Hofmann,7161.0,2025/2026,Bundesliga,23,1009,2.0,1.71,0.29,23.0,...,96.58,23.07,0.27,0.00,0.45,0.18,NaN,NaN,NaN,NaN
8,Carlos Domínguez,7689.0,2025/2026,"La Liga, Europa League",11,724,0.0,0.55,-0.55,3.0,...,122.23,80.45,0.00,0.00,0.00,0.00,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4742,Jan Virgili,1301775.0,2025/2026,La Liga,31,1797,2.0,2.33,-0.33,32.0,...,176.82,101.62,0.90,0.05,0.40,0.05,NaN,NaN,NaN,NaN
4744,Nathan Mbala,1307004.0,2025/2026,Ligue 1,13,436,2.0,0.82,1.18,15.0,...,130.91,97.49,0.83,0.21,1.03,0.21,NaN,NaN,NaN,NaN
4751,Christian Kofane,1364454.0,2025/2026,"Bundesliga, Champions League",40,1835,6.0,9.60,-3.60,60.0,...,55.63,30.36,0.59,0.00,0.29,0.15,NaN,NaN,NaN,NaN
4752,Thiago Pitarch,1375227.0,2025/2026,"La Liga, Champions League",16,816,0.0,0.41,-0.41,6.0,...,170.11,31.61,0.00,0.00,0.11,0.11,NaN,NaN,NaN,NaN


In [19]:
data_opta_analyst_agg.to_csv("data_opta_analyst_agg.csv",index=False)

In [20]:
#data_jugadores.to_csv("data_jugadores.csv",index=False)

### Sofascore

In [21]:
# On ajoute l'identifiant de fotmob afin d'associer ces données à celle d'opta

data_complete = data_complete.merge(data_jugadores[["tm_id", "sofascore_id"]],on="tm_id",how="left")

data_complete["sofascore_id"] = data_complete["sofascore_id"].fillna("")

# On analyse le nombre de lignes où sofascore_id est manquant
print((data_complete["sofascore_id"] == "").sum())

88


In [22]:
# On récupère les joueurs sans sofascore_id

missing_sofa = data_complete[data_complete["sofascore_id"].isna() | (data_complete["sofascore_id"] == "")].copy()

# Liste des noms disponibles dans data_sofascore
sofa_names = data_sofascore["name"].dropna().unique().tolist()

# Fonction de fuzzy matching

def get_sofascore_match(player_name):

    if pd.isna(player_name):
        return pd.Series([None, None, None])

    match = process.extractOne(player_name,sofa_names,scorer=fuzz.ratio)

    if match is None:
        return pd.Series([None, None, None])

    matched_name, score, _ = match

    # On conserve uniquement les correspondances >= 85 %
    if score >= 85:

        player_id = data_sofascore.loc[data_sofascore["name"] == matched_name,"player_id"].iloc[0]

        return pd.Series([player_id,matched_name,score])

    return pd.Series([None,None,score])

# Fuzzy matching

missing_sofa[["sofascore_id_fuzzy", "sofascore_name_match", "match_score"]] = missing_sofa["name"].apply(get_sofascore_match)

# On ajoute les sofascore_id trouvés dans data_complete

data_complete.loc[missing_sofa.index,"sofascore_id"] = missing_sofa["sofascore_id_fuzzy"]

# Correspondances manuelles pour les joueurs restants
manual_sofascore_ids = {
    "Mohamed Meïté": 1606642, "Beraldo": 1108441, "Milan Djuric": 76132, "Copete": 913695, "Al Musrati": 868958,
    "Lee Jae-Sung": 537552, "CJ Egan-Riley" :1131448, "Lionel Mpasi": 599192, "Alex Amorim" : 1511706, "Djené Dakonam" : 307702, 
    "Hákon Haraldsson":1138804, "Ibrahim Sulemana":105905, "Djordje Petrovic":882604,"Étienne Youté":980406}

# Ajout des sofascore_id manuels
mask_manual = data_complete["name"].isin(manual_sofascore_ids)

data_complete.loc[mask_manual,"sofascore_id"] = data_complete.loc[mask_manual,"name"].map(manual_sofascore_ids)

# Nombre de lignes restantes sans sofascore_id

mask_missing_final = (data_complete["sofascore_id"].isna() |
    (data_complete["sofascore_id"] == ""))

print("Nombre de lignes restantes sans sofascore_id :",mask_missing_final.sum())

# Liste des joueurs restant sans sofascore_id
missing_sofascore_final = data_complete.loc[
    mask_missing_final,["name","tm_id"]
].copy()

display(missing_sofascore_final)

# On enlève les lignes sans sofascore_id
data_complete = data_complete[
    data_complete["sofascore_id"].notna() &
    (data_complete["sofascore_id"] != "")
].copy()

# Réinitialisation de l'index
data_complete = data_complete.reset_index(drop=True)

print("Nombre de lignes restantes dans data_complete :",len(data_complete))

Nombre de lignes restantes sans sofascore_id : 5


,name,tm_id
1131,Carlos Protesoni,431190.0
1694,Urko González de Zárate,625651.0
2251,Jair Cunha,984096.0
2252,Jair Cunha,984096.0
2341,Pablo,1100071.0


Nombre de lignes restantes dans data_complete : 2402


In [23]:
# On filtre les données du big 5 + compétitions européennes
data_sofascore = data_sofascore[data_sofascore["league"].isin(top_leagues)].copy()

# Prétraitement de la saison
def extract_season(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    match = re.search(r"(\d{2})/(\d{2})", value)

    if match:
        start = match.group(1)
        end = match.group(2)

        return f"20{start}/20{end}"

    match_year = re.search(r"\b(20\d{2})\b", value)

    if match_year:
        return match_year.group(1)

    return np.nan

data_sofascore["season"] = data_sofascore["season"].apply(extract_season)

# Uniformisation des identifiants
data_complete["sofascore_id"] = data_complete["sofascore_id"].astype(str).str.replace(".0", "", regex=False).str.strip()

data_sofascore["player_id"] = data_sofascore["player_id"].astype(str).str.replace(".0", "", regex=False).str.strip()

# Variables à sommer
sum_cols_sofascore = ["stat_bigChancesCreated","stat_bigChancesMissed","stat_keyPasses","stat_penaltiesTaken","stat_offsides"]

# Variables déjà calculées par 90 minutes
weighted_cols_sofascore = ["stat_km_per90","stat_sprints_per90"]

# Variable où l'on garde la valeur maximale
max_cols_sofascore = ["stat_topSpeed"]

# Conversion en numérique
all_stats_sofascore = (sum_cols_sofascore+ weighted_cols_sofascore+ max_cols_sofascore)

for col in all_stats_sofascore:
    data_sofascore[col] = pd.to_numeric(data_sofascore[col],errors="coerce")

# Conversion des minutes SofaScore
data_sofascore["stat_minutesPlayed"] = pd.to_numeric(data_sofascore["stat_minutesPlayed"],errors="coerce")

# Somme des statistiques de volume par joueur et saison
data_sofascore_sum = data_sofascore.groupby(["player_id","season"],as_index=False)[sum_cols_sofascore].sum(min_count=1)

# Fonction de moyenne pondérée par les minutes SofaScore
def weighted_average(group,col):
    valid = (group[col].notna() & group["stat_minutesPlayed"].notna() & (group["stat_minutesPlayed"] > 0))

    if valid.sum() == 0:
        return np.nan

    return np.average(group.loc[valid,col],weights=group.loc[valid,"stat_minutesPlayed"])

# Moyenne pondérée des statistiques déjà en per90
weighted_results = []

for (player_id,season),group in data_sofascore.groupby(["player_id","season"]):
    row = {"player_id": player_id,"season": season}

    for col in weighted_cols_sofascore:
        row[col] = weighted_average(group,col)

    weighted_results.append(row)

data_sofascore_weighted = pd.DataFrame(weighted_results)

# Top speed : maximum par joueur et saison
data_sofascore_max = data_sofascore.groupby(["player_id","season"],as_index=False)[max_cols_sofascore].max()

# Regroupement de toutes les statistiques
data_sofascore_agg = data_sofascore_sum.merge(data_sofascore_weighted,on=["player_id","season"],how="outer").merge(
    data_sofascore_max,on=["player_id","season"],how="outer")

# Correction manuelle de certains sofascore_id
manual_sofascore_ids = {
    "Abdoulaye Faye": "1529051", "Álex Moreno": "294593", "Álvaro García": "345111", "Ameen Al Dakhil": "1979970",
    "Amine Sbaï": "1392829", "André Almeida": "845693", "André Silva": "190159", "Angel Gomes": "867441", "Antonín Kinsky": "1031251",
    "Boubacar Traoré": "982613", "Bruno Fernandes": "288205", "Callum Wilson": "113956", "Daniel Svensson": "1021272","Endrick": "1174937",
    "Estêvão": "1597265", "Ibrahim Sangaré": "843754", "Ismaël Koné": "1134351", "Javi Guerra": "1122610", "Jesús Rodríguez": "1800245",
    "Johan Vásquez": "889785", "Juan Cruz": "814360", "Juan Rodríguez": "1485391", "Karim Coulibaly": "1797852", "Lewis Cook": "548188",
    "Luis Díaz": "883537", "Luiz Felipe": "850035", "Malick Fofana": "1195784", "Mamadou Coulibaly": "1410148", "Marcus Pedersen": "934409",
    "Mohamed Bamba": "1405239", "Ousmane Camara": "1049538", "Pablo Ibáñez": "1084381", "Pablo Martínez": "927066", "Rodrigo Muniz": "1015256",
    "Rodrigo Ribeiro": "1215904", "Rodrigo Riquelme": "989113", "Sergio Herrera": "294377", "Thomas Kristensen": "1063373",
    "Jacob Ondrejka" : "1004790"}

# On remplace le sofascore_id uniquement pour les joueurs renseignés manuellement
mask_manual_sofascore = data_complete["name"].isin(manual_sofascore_ids)

data_complete.loc[mask_manual_sofascore,"sofascore_id"] = data_complete.loc[mask_manual_sofascore,"name"].map(manual_sofascore_ids)

# Jointure avec data_complete
data_complete = data_complete.merge(data_sofascore_agg,left_on=["sofascore_id","season"],right_on=["player_id","season"],how="left")

# Suppression de player_id ajouté par la jointure
data_complete = data_complete.drop(columns="player_id")

# Conversion des minutes Opta présentes dans data_complete
data_complete["minutes"] = pd.to_numeric(data_complete["minutes"],errors="coerce")

# Calcul des statistiques par 90 minutes à partir des minutes de data_complete
data_complete["stat_bigChancesCreated_per90"] = np.where(data_complete["minutes"] > 0,
    data_complete["stat_bigChancesCreated"] / data_complete["minutes"] * 90,np.nan)

data_complete["stat_bigChancesMissed_per90"] = np.where(data_complete["minutes"] > 0,
    data_complete["stat_bigChancesMissed"] / data_complete["minutes"] * 90,np.nan)

data_complete["stat_keyPasses_per90"] = np.where(data_complete["minutes"] > 0,
    data_complete["stat_keyPasses"] / data_complete["minutes"] * 90,np.nan)

# Liste des statistiques SofaScore finales
sofascore_cols_final = ["stat_bigChancesCreated","stat_bigChancesCreated_per90","stat_bigChancesMissed","stat_bigChancesMissed_per90",
    "stat_keyPasses","stat_keyPasses_per90","stat_penaltiesTaken","stat_offsides","stat_km_per90","stat_sprints_per90","stat_topSpeed"]

# Vérifications finales
print("\nNombre de lignes dans data_complete :",len(data_complete))

# Lignes sans aucune statistique SofaScore
mask_no_sofascore_stats = data_complete[sofascore_cols_final].isna().all(axis=1)

print("\nNombre de lignes sans aucune statistique SofaScore :",mask_no_sofascore_stats.sum())

# On enlève les lignes sans aucune statistique SofaScore
data_complete = data_complete[~mask_no_sofascore_stats].copy()

# Réinitialisation de l'index
data_complete = data_complete.reset_index(drop=True)

print("Nombre de lignes restantes dans data_complete :",len(data_complete))

# On arrondit toutes les variables numériques à 2 chiffres après la virgule
cols_numeric = data_complete.select_dtypes(include="number").columns

data_complete[cols_numeric] = data_complete[cols_numeric].round(2)


Nombre de lignes dans data_complete : 2402

Nombre de lignes sans aucune statistique SofaScore : 14
Nombre de lignes restantes dans data_complete : 2388


In [24]:
# On récupère les lignes sans aucune statistique SofaScore
data_sans_sofascore_stats = data_complete[mask_no_sofascore_stats].copy()

# Export en CSV
data_sans_sofascore_stats.to_csv("data_sans_sofascore_stats.csv",index=False,encoding="utf-8-sig")


/var/folders/9j/n_pg7t817pn01v85cchv_b3r0000gn/T/ipykernel_15989/2598528339.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data_sans_sofascore_stats = data_complete[mask_no_sofascore_stats].copy()


In [12]:
data_complete.to_csv("data_complete.csv",index=False)

### Fotmob

In [ ]:
# On ajoute l'identifiant de fotmob afin d'associer ces données à celle d'opta

data_complete = data_complete.merge(data_jugadores[["tm_id", "fotmob_id"]],on="tm_id",how="left")

data_complete["fotmob_id"] = data_complete["fotmob_id"].fillna("")

# On analyse le nombre de lignes où fotmob_id est manquant
print((data_complete["fotmob_id"] == "").sum())